# 11 MMseqs-Enhanced Ceiling-Aware Weighting


## Strategies Compared

This notebook evaluates:

1. `best_individual_test_oracle`: test-set oracle upper reference, not deployable.
2. `same_dataset_head`: one head trained on the target dataset.
3. `validation_selected_single_head`: best validation head per target.
4. `simple_mean_all_heads`: unweighted mean of all heads.
5. `exact_ceiling_weighted`: validation headroom weighted by exact annotation ceilings.
6. `mmseqs95_ceiling_weighted`: validation headroom weighted by MMseqs 95% identity ceilings.
7. `mmseqs80_ceiling_weighted`: validation headroom weighted by MMseqs 80% identity ceilings.
8. `blended_exact_mmseqs_bio_weighted`: reliability-weighted blend of exact, MMseqs, and biological-family priors.

The weighted strategies include a same-dataset anchor so the biological prior can add compatible heads without erasing the target-specific annotation definition.


In [ ]:
from __future__ import annotations
import json
import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
UDONPRED_DIR = ROOT / "UdonPred"
RESULTS = ROOT / "results"
TEST_PREDICTION_ROOT = RESULTS / "udonpred_matrix" / "predictions"
VALID_PREDICTION_ROOT = RESULTS / "udonpred_validation_matrix" / "predictions"
OUT_DIR = RESULTS / "mmseqs_ceiling_aware_weighting"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]
NEGATED_DATASETS = {"chezod", "plddt"}
MASK_VALUE = 999.0
METRIC_COLS = [
    "trizod",
    "chezod",
    "softdis",
    "pdbflex",
    "atlas",
    "plddt",
    "disprot\n(AP)",
    "disprot\n(AUROC)",
]
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", context="notebook")


## 1. Load Predictions and Evaluation Helpers


In [ ]:
def read_jsonl_records(path: Path) -> dict[str, dict[str, object]]:
    records = {}
    with path.open() as handle:
        for line in handle:
            raw = json.loads(line)
            records[str(raw["id"])] = {
                "sequence": str(raw["x_0"]),
                "labels": np.asarray(raw["y"], dtype=np.float64),
            }
    return records


def normalize_prediction_id(protein_id: str) -> str:
    protein_id = protein_id.strip().lstrip(">")
    if len(protein_id) >= 6 and protein_id.isdigit():
        base_len = len(protein_id) - 3
        return protein_id[:base_len] + "_" + "_".join(protein_id[base_len:])
    return protein_id


def read_caid_dir(input_dir: Path) -> dict[str, np.ndarray]:
    predictions = {}
    for path in sorted(input_dir.glob("*.caid")):
        with path.open() as handle:
            lines = handle.readlines()
        if not lines:
            raise ValueError(f"Empty prediction file: {path}")
        protein_id = normalize_prediction_id(lines[0])
        scores = []
        for line in lines[1:]:
            line = line.strip()
            if line:
                scores.append(float(line.split("\t")[2]))
        predictions[protein_id] = np.asarray(scores, dtype=np.float64)
    return predictions


def labels_in_disorder_direction(labels: np.ndarray, dataset: str) -> np.ndarray:
    return -labels if dataset in NEGATED_DATASETS else labels


def preds_in_disorder_direction(preds: np.ndarray, train_dataset: str) -> np.ndarray:
    return -preds if train_dataset in NEGATED_DATASETS else preds


def metric_columns_for_dataset(dataset: str) -> list[str]:
    return ["disprot\n(AP)", "disprot\n(AUROC)"] if dataset == "disprot" else [dataset]


def primary_metric(dataset: str) -> str:
    return "disprot\n(AP)" if dataset == "disprot" else dataset


def metric_target_dataset(metric: str) -> str:
    return "disprot" if metric.startswith("disprot") else metric


def evaluate_vector(labels: np.ndarray, preds: np.ndarray, dataset: str) -> dict[str, float]:
    if len(labels) == 0:
        return {column: math.nan for column in metric_columns_for_dataset(dataset)}
    if dataset == "disprot":
        binary_labels = labels.astype(int)
        if np.unique(binary_labels).size < 2:
            return {"disprot\n(AP)": math.nan, "disprot\n(AUROC)": math.nan}
        return {
            "disprot\n(AP)": float(average_precision_score(binary_labels, preds)),
            "disprot\n(AUROC)": float(roc_auc_score(binary_labels, preds)),
        }
    if np.unique(labels).size < 2 or np.unique(preds).size < 2:
        return {dataset: math.nan}
    return {dataset: float(spearmanr(preds, labels).statistic)}


def load_aligned_stack(split: str, prediction_root: Path, target_dataset: str) -> dict[str, object]:
    records = read_jsonl_records(UDONPRED_DIR / "data" / target_dataset / f"{split}.jsonl")
    pred_maps = {
        train_dataset: read_caid_dir(prediction_root / f"{train_dataset}_{target_dataset}")
        for train_dataset in DATASETS
    }

    label_chunks = []
    pred_chunks_by_train = {train_dataset: [] for train_dataset in DATASETS}
    residue_count = 0
    protein_count = 0
    for protein_id, record in records.items():
        labels = np.asarray(record["labels"], dtype=np.float64)
        mask = np.isfinite(labels) & (labels != MASK_VALUE)
        if not np.any(mask):
            continue

        for train_dataset, pred_map in pred_maps.items():
            if protein_id not in pred_map:
                raise ValueError(f"Missing prediction for {protein_id} in {train_dataset}_{target_dataset}")
            preds = pred_map[protein_id]
            if len(preds) != len(labels):
                raise ValueError(
                    f"{train_dataset}_{target_dataset}/{protein_id}: prediction length {len(preds)} "
                    f"!= label length {len(labels)}"
                )
            pred_chunks_by_train[train_dataset].append(preds_in_disorder_direction(preds, train_dataset)[mask])

        label_chunks.append(labels_in_disorder_direction(labels, target_dataset)[mask])
        residue_count += int(mask.sum())
        protein_count += 1

    y = np.concatenate(label_chunks)
    X = np.column_stack([np.concatenate(pred_chunks_by_train[train_dataset]) for train_dataset in DATASETS])
    return {"dataset": target_dataset, "split": split, "y": y, "X": X, "n_residues": residue_count, "n_proteins": protein_count}


def score_individual_heads(stacks: dict[str, dict[str, object]]) -> pd.DataFrame:
    rows = []
    for train_index, train_dataset in enumerate(DATASETS):
        row = {"train_dataset": train_dataset}
        for target_dataset, stack in stacks.items():
            row.update(evaluate_vector(stack["y"], stack["X"][:, train_index], target_dataset))
        rows.append(row)
    return pd.DataFrame(rows).set_index("train_dataset")[METRIC_COLS]


In [ ]:
test_stacks = {dataset: load_aligned_stack("test", TEST_PREDICTION_ROOT, dataset) for dataset in DATASETS}
valid_stacks = {dataset: load_aligned_stack("valid", VALID_PREDICTION_ROOT, dataset) for dataset in DATASETS}

stack_summary = pd.DataFrame(
    [
        {"split": split, "dataset": dataset, "n_proteins": stack["n_proteins"], "n_residues": stack["n_residues"]}
        for split, stacks in [("test", test_stacks), ("valid", valid_stacks)]
        for dataset, stack in stacks.items()
    ]
)
display(stack_summary)
stack_summary.to_csv(OUT_DIR / "stack_summary.csv", index=False)


## 2. Baseline Scores and Individual Heads


In [ ]:
def load_best_simple_baseline_scores() -> pd.Series:
    path = RESULTS / "normalized_headroom" / "best_simple_baseline_per_metric.csv"
    if path.exists():
        summary = pd.read_csv(path)
        return summary.set_index("test_metric")["best_simple_baseline_score"].reindex(METRIC_COLS).apply(pd.to_numeric, errors="coerce")

    baseline_matrix = pd.read_csv(RESULTS / "simple_baselines" / "matrix.csv")
    baseline_scores = baseline_matrix.set_index(["baseline", "train_dataset"])[METRIC_COLS]
    return baseline_scores.apply(pd.to_numeric, errors="coerce").max(axis=0)


valid_individual = score_individual_heads(valid_stacks)
test_individual = score_individual_heads(test_stacks)
best_individual_scores = test_individual.max(axis=0)
best_individual_heads = test_individual.idxmax(axis=0)
best_simple_baseline_scores = load_best_simple_baseline_scores()

validation_selected_head_by_target = {
    dataset: valid_individual[primary_metric(dataset)].idxmax()
    for dataset in DATASETS
}

same_dataset_weights = {
    dataset: pd.Series({head: 1.0 if head == dataset else 0.0 for head in DATASETS}, dtype=float)
    for dataset in DATASETS
}
validation_selected_weights = {
    dataset: pd.Series({head: 1.0 if head == validation_selected_head_by_target[dataset] else 0.0 for head in DATASETS}, dtype=float)
    for dataset in DATASETS
}

valid_individual.to_csv(OUT_DIR / "validation_individual_matrix.csv")
test_individual.to_csv(OUT_DIR / "test_individual_matrix.csv")

display(test_individual.style.format("{:.3f}"))
display(pd.DataFrame({"validation_selected_head": validation_selected_head_by_target, "best_test_head_oracle": {metric_target_dataset(m): h for m, h in best_individual_heads.items() if not m.startswith('disprot') or m.endswith('(AP)')}}))


## 3. Build Exact, MMseqs, and Biological Compatibility Matrices


In [ ]:
ANNOTATION_FAMILY = {
    "trizod": "NMR chemical-shift disorder",
    "chezod": "NMR chemical-shift disorder",
    "softdis": "derived soft disorder",
    "atlas": "derived soft disorder",
    "plddt": "AlphaFold confidence proxy",
    "disprot": "curated binary disorder",
    "pdbflex": "structural flexibility/dynamics",
}

HIGH_EXPECTED = {
    tuple(sorted(pair))
    for pair in [
        ("trizod", "chezod"),
        ("softdis", "plddt"),
        ("softdis", "disprot"),
        ("chezod", "plddt"),
    ]
}
MEDIUM_EXPECTED = {
    tuple(sorted(pair))
    for pair in [
        ("trizod", "softdis"),
        ("trizod", "plddt"),
        ("chezod", "softdis"),
        ("atlas", "softdis"),
        ("atlas", "plddt"),
        ("atlas", "chezod"),
    ]
}
LOW_EXPECTED = {
    tuple(sorted(pair))
    for pair in [
        ("pdbflex", "trizod"),
        ("pdbflex", "chezod"),
        ("pdbflex", "softdis"),
        ("pdbflex", "atlas"),
        ("pdbflex", "plddt"),
        ("pdbflex", "disprot"),
    ]
}


def expected_similarity(a: str, b: str) -> str:
    key = tuple(sorted((a, b)))
    if a == b:
        return "same_dataset"
    if key in HIGH_EXPECTED:
        return "high"
    if key in MEDIUM_EXPECTED:
        return "medium"
    if key in LOW_EXPECTED:
        return "low"
    return "unknown"


def biological_prior_value(a: str, b: str) -> float:
    if a == b:
        return 1.0
    expectation = expected_similarity(a, b)
    if expectation == "high":
        return 0.75
    if expectation == "medium":
        return 0.45
    if expectation == "low":
        return 0.08
    if ANNOTATION_FAMILY[a] == ANNOTATION_FAMILY[b]:
        return 0.60
    return 0.20


def overlap_confidence(n_residues: float) -> float:
    if pd.isna(n_residues) or n_residues <= 0:
        return 0.0
    if n_residues > 10_000:
        return 1.0
    if n_residues >= 1_000:
        return 0.8
    if n_residues >= 300:
        return 0.6
    return 0.3


def source_reliability(level: str, min_identity: float | None, n_residues: float) -> float:
    if level == "exact":
        base = 1.0
    elif pd.notna(min_identity):
        if min_identity >= 98:
            base = 0.90
        elif min_identity >= 95:
            base = 0.85
        elif min_identity >= 90:
            base = 0.75
        elif min_identity >= 85:
            base = 0.68
        else:
            base = 0.60
    else:
        base = 0.25
    return base * overlap_confidence(n_residues)


def metric_for_pair_and_target(dataset_a: str, dataset_b: str, target_metric: str) -> str:
    target_dataset = metric_target_dataset(target_metric)
    if dataset_a == "disprot" or dataset_b == "disprot":
        return "average_precision" if target_metric == "disprot\n(AP)" else "auroc"
    return "spearman"


def build_compatibility_from_summary(summary: pd.DataFrame, comparison_level: str, target_metric: str) -> pd.Series:
    target_dataset = metric_target_dataset(target_metric)
    values = pd.Series(0.0, index=DATASETS, dtype=float)
    values[target_dataset] = 1.0
    for train_dataset in DATASETS:
        if train_dataset == target_dataset:
            continue
        metric = metric_for_pair_and_target(train_dataset, target_dataset, target_metric)
        pair = summary[
            (summary["comparison_level"] == comparison_level)
            & (
                ((summary["dataset_a"] == train_dataset) & (summary["dataset_b"] == target_dataset))
                | ((summary["dataset_a"] == target_dataset) & (summary["dataset_b"] == train_dataset))
            )
            & (summary["metric"] == metric)
        ]
        if pair.empty:
            values[train_dataset] = np.nan
        else:
            values[train_dataset] = float(pd.to_numeric(pair.iloc[0]["value"], errors="coerce"))
    return values.clip(lower=0.0, upper=1.0)


def compatibility_matrix_from_summary(summary: pd.DataFrame, comparison_level: str) -> pd.DataFrame:
    return pd.DataFrame(
        {metric: build_compatibility_from_summary(summary, comparison_level, metric) for metric in METRIC_COLS}
    ).reindex(index=DATASETS, columns=METRIC_COLS)


mmseqs_summary = pd.read_csv(RESULTS / "annotation_ceiling_mmseqs" / "annotation_ceiling_summary.csv")
mmseqs_summary["value"] = pd.to_numeric(mmseqs_summary["value"], errors="coerce")
mmseqs_summary["min_identity"] = pd.to_numeric(mmseqs_summary["min_identity"], errors="coerce")
mmseqs_summary["n_residues_compared"] = pd.to_numeric(mmseqs_summary["n_residues_compared"], errors="coerce")

exact_compat = compatibility_matrix_from_summary(mmseqs_summary, "exact")
mmseqs95_compat = compatibility_matrix_from_summary(mmseqs_summary, "mmseqs_95")
mmseqs80_compat = compatibility_matrix_from_summary(mmseqs_summary, "mmseqs_80")
bio_prior_compat = pd.DataFrame(
    {
        metric: pd.Series({train: biological_prior_value(train, metric_target_dataset(metric)) for train in DATASETS})
        for metric in METRIC_COLS
    }
).reindex(index=DATASETS, columns=METRIC_COLS)

exact_compat.to_csv(OUT_DIR / "exact_compatibility_matrix.csv")
mmseqs95_compat.to_csv(OUT_DIR / "mmseqs95_compatibility_matrix.csv")
mmseqs80_compat.to_csv(OUT_DIR / "mmseqs80_compatibility_matrix.csv")
bio_prior_compat.to_csv(OUT_DIR / "biological_prior_compatibility_matrix.csv")

display(exact_compat.style.format("{:.3f}"))
display(mmseqs95_compat.style.format("{:.3f}"))
display(bio_prior_compat.style.format("{:.3f}"))


## 4. Reliability-Blended Compatibility


In [ ]:
def observed_pair_value(summary: pd.DataFrame, train_dataset: str, target_metric: str, comparison_level: str) -> tuple[float, float, float]:
    target_dataset = metric_target_dataset(target_metric)
    if train_dataset == target_dataset:
        return 1.0, 1.0, math.inf
    metric = metric_for_pair_and_target(train_dataset, target_dataset, target_metric)
    pair = summary[
        (summary["comparison_level"] == comparison_level)
        & (
            ((summary["dataset_a"] == train_dataset) & (summary["dataset_b"] == target_dataset))
            | ((summary["dataset_a"] == target_dataset) & (summary["dataset_b"] == train_dataset))
        )
        & (summary["metric"] == metric)
    ]
    if pair.empty:
        return np.nan, 0.0, np.nan
    row = pair.iloc[0]
    value = float(pd.to_numeric(row["value"], errors="coerce"))
    n_residues = float(pd.to_numeric(row["n_residues_compared"], errors="coerce"))
    min_identity = float(pd.to_numeric(row["min_identity"], errors="coerce")) if pd.notna(row.get("min_identity")) else np.nan
    reliability = source_reliability(comparison_level, min_identity, n_residues)
    return max(0.0, min(1.0, value)) if pd.notna(value) else np.nan, reliability, n_residues


def blended_compatibility_for_metric(metric: str) -> pd.Series:
    values = {}
    for train_dataset in DATASETS:
        target_dataset = metric_target_dataset(metric)
        if train_dataset == target_dataset:
            values[train_dataset] = 1.0
            continue

        evidence = []
        for level in ["exact", "mmseqs_98", "mmseqs_95", "mmseqs_90", "mmseqs_85", "mmseqs_80"]:
            value, reliability, _ = observed_pair_value(mmseqs_summary, train_dataset, metric, level)
            if pd.notna(value) and reliability > 0:
                evidence.append((value, reliability))

        bio_value = biological_prior_value(train_dataset, target_dataset)
        # Biological prior is deliberately low weight: it regularizes missing/sparse measured evidence.
        evidence.append((bio_value, 0.25))
        numerator = sum(value * weight for value, weight in evidence)
        denominator = sum(weight for _, weight in evidence)
        values[train_dataset] = numerator / denominator if denominator else bio_value
    return pd.Series(values, dtype=float).clip(lower=0.0, upper=1.0)


blended_compat = pd.DataFrame({metric: blended_compatibility_for_metric(metric) for metric in METRIC_COLS}).reindex(index=DATASETS, columns=METRIC_COLS)
blended_compat.to_csv(OUT_DIR / "blended_exact_mmseqs_bio_compatibility_matrix.csv")
display(blended_compat.style.format("{:.3f}"))


## 5. Convert Compatibility Into Weights


In [ ]:
def validation_headroom_signal(target_dataset: str) -> pd.Series:
    metric = primary_metric(target_dataset)
    validation_scores = valid_individual[metric].reindex(DATASETS).astype(float)
    baseline_score = float(best_simple_baseline_scores[metric])
    signal = (validation_scores - baseline_score).clip(lower=0.0).fillna(0.0)
    if signal.sum() <= 1e-12:
        signal.loc[target_dataset] = 1.0
    return signal


def normalize_weights(raw: pd.Series) -> pd.Series:
    raw = raw.reindex(DATASETS).fillna(0.0).clip(lower=0.0).astype(float)
    if raw.sum() <= 1e-12:
        raw[:] = 0.0
        return raw
    return raw / raw.sum()


def anchored_weights(raw: pd.Series, target_dataset: str, self_min_weight: float) -> pd.Series:
    norm = normalize_weights(raw)
    one_hot = pd.Series({dataset: 1.0 if dataset == target_dataset else 0.0 for dataset in DATASETS}, dtype=float)
    if self_min_weight <= 0:
        return norm
    return (1.0 - self_min_weight) * norm + self_min_weight * one_hot


def compatibility_weighted_strategy(compat: pd.DataFrame, target_dataset: str, self_min_weight: float = 0.45, gamma: float = 2.0) -> pd.Series:
    metric = primary_metric(target_dataset)
    signal = validation_headroom_signal(target_dataset)
    compatible = compat[metric].reindex(DATASETS).fillna(0.0).clip(lower=0.0, upper=1.0)
    raw = signal * np.power(compatible, gamma)
    return anchored_weights(raw, target_dataset, self_min_weight=self_min_weight)


weight_strategies = {
    "same_dataset_head": same_dataset_weights,
    "validation_selected_single_head": validation_selected_weights,
    "exact_ceiling_weighted": {
        dataset: compatibility_weighted_strategy(exact_compat, dataset, self_min_weight=0.45, gamma=2.0)
        for dataset in DATASETS
    },
    "mmseqs95_ceiling_weighted": {
        dataset: compatibility_weighted_strategy(mmseqs95_compat, dataset, self_min_weight=0.45, gamma=2.0)
        for dataset in DATASETS
    },
    "mmseqs80_ceiling_weighted": {
        dataset: compatibility_weighted_strategy(mmseqs80_compat, dataset, self_min_weight=0.45, gamma=2.0)
        for dataset in DATASETS
    },
    "blended_exact_mmseqs_bio_weighted": {
        dataset: compatibility_weighted_strategy(blended_compat, dataset, self_min_weight=0.55, gamma=2.0)
        for dataset in DATASETS
    },
}

weight_rows = []
for strategy, by_dataset in weight_strategies.items():
    for target_dataset, weights in by_dataset.items():
        weight_rows.append({"strategy": strategy, "target_dataset": target_dataset, **weights.to_dict()})
weights_df = pd.DataFrame(weight_rows)
weights_df.to_csv(OUT_DIR / "strategy_weights.csv", index=False)

display(weights_df.style.format({dataset: "{:.3f}" for dataset in DATASETS}))


## 6. Evaluate Strategies on Held-Out Test Sets


In [ ]:
def predictions_for_weights(weights: pd.Series, target_dataset: str, stack: dict[str, object]) -> np.ndarray:
    w = weights.reindex(DATASETS).fillna(0.0).to_numpy(dtype=np.float64)
    return stack["X"] @ w


def evaluate_strategy_matrix() -> pd.DataFrame:
    rows = []
    oracle_row = {"strategy": "best_individual_test_oracle", **best_individual_scores.to_dict()}
    rows.append(oracle_row)

    mean_row = {"strategy": "simple_mean_all_heads"}
    for dataset, stack in test_stacks.items():
        preds = stack["X"].mean(axis=1)
        mean_row.update(evaluate_vector(stack["y"], preds, dataset))
    rows.append(mean_row)

    for strategy, by_dataset in weight_strategies.items():
        row = {"strategy": strategy}
        for dataset, stack in test_stacks.items():
            preds = predictions_for_weights(by_dataset[dataset], dataset, stack)
            row.update(evaluate_vector(stack["y"], preds, dataset))
        rows.append(row)
    return pd.DataFrame(rows).set_index("strategy")[METRIC_COLS]


strategy_matrix = evaluate_strategy_matrix()
strategy_delta_vs_best_individual = strategy_matrix.subtract(best_individual_scores, axis="columns")
strategy_delta_vs_same = strategy_matrix.subtract(strategy_matrix.loc["same_dataset_head"], axis="columns")
strategy_delta_vs_validation_selected = strategy_matrix.subtract(strategy_matrix.loc["validation_selected_single_head"], axis="columns")

available_headroom = 1.0 - best_simple_baseline_scores
raw_headroom = strategy_matrix.subtract(best_simple_baseline_scores, axis="columns")
normalized_headroom = raw_headroom.divide(available_headroom, axis="columns")
normalized_headroom.loc[:, available_headroom <= 1e-12] = np.nan

summary_rows = []
for strategy in strategy_matrix.index:
    deltas_best = strategy_delta_vs_best_individual.loc[strategy]
    deltas_same = strategy_delta_vs_same.loc[strategy]
    deltas_validation = strategy_delta_vs_validation_selected.loc[strategy]
    summary_rows.append({
        "strategy": strategy,
        "mean_score": float(strategy_matrix.loc[strategy].mean()),
        "mean_normalized_headroom": float(normalized_headroom.loc[strategy].mean()),
        "mean_delta_vs_best_individual": float(deltas_best.mean()),
        "mean_delta_vs_same_dataset_head": float(deltas_same.mean()),
        "mean_delta_vs_validation_selected": float(deltas_validation.mean()),
        "wins_vs_best_individual": int((deltas_best > 1e-6).sum()),
        "ties_vs_best_individual": int((deltas_best.abs() <= 1e-6).sum()),
        "losses_vs_best_individual": int((deltas_best < -1e-6).sum()),
        "wins_vs_same_dataset_head": int((deltas_same > 1e-6).sum()),
        "wins_vs_validation_selected": int((deltas_validation > 1e-6).sum()),
        "worst_delta_vs_same_dataset_head": float(deltas_same.min()),
    })
strategy_summary = pd.DataFrame(summary_rows).set_index("strategy")

strategy_matrix.to_csv(OUT_DIR / "strategy_matrix.csv")
strategy_delta_vs_best_individual.to_csv(OUT_DIR / "strategy_delta_vs_best_individual.csv")
strategy_delta_vs_same.to_csv(OUT_DIR / "strategy_delta_vs_same_dataset_head.csv")
strategy_delta_vs_validation_selected.to_csv(OUT_DIR / "strategy_delta_vs_validation_selected.csv")
normalized_headroom.to_csv(OUT_DIR / "strategy_normalized_headroom.csv")
strategy_summary.to_csv(OUT_DIR / "strategy_summary.csv")

display(strategy_matrix.style.format("{:.3f}"))
display(normalized_headroom.style.format("{:.3f}"))
display(strategy_summary.style.format("{:.3f}"))


## 7. Per-Metric Winners and Diagnostics


In [ ]:
non_oracle = strategy_matrix.drop(index="best_individual_test_oracle")
winner_rows = []
for metric in METRIC_COLS:
    best_strategy = non_oracle[metric].idxmax()
    winner_rows.append({
        "metric": metric,
        "best_non_oracle_strategy": best_strategy,
        "best_non_oracle_score": float(non_oracle.loc[best_strategy, metric]),
        "best_individual_oracle_score": float(strategy_matrix.loc["best_individual_test_oracle", metric]),
        "same_dataset_score": float(strategy_matrix.loc["same_dataset_head", metric]),
        "validation_selected_score": float(strategy_matrix.loc["validation_selected_single_head", metric]),
        "delta_best_non_oracle_vs_same": float(non_oracle.loc[best_strategy, metric] - strategy_matrix.loc["same_dataset_head", metric]),
        "delta_best_non_oracle_vs_validation": float(non_oracle.loc[best_strategy, metric] - strategy_matrix.loc["validation_selected_single_head", metric]),
    })
winners = pd.DataFrame(winner_rows)
winners.to_csv(OUT_DIR / "best_non_oracle_strategy_by_metric.csv", index=False)
display(winners.style.format({
    "best_non_oracle_score": "{:.3f}",
    "best_individual_oracle_score": "{:.3f}",
    "same_dataset_score": "{:.3f}",
    "validation_selected_score": "{:.3f}",
    "delta_best_non_oracle_vs_same": "{:+.3f}",
    "delta_best_non_oracle_vs_validation": "{:+.3f}",
}))


## 8. Plots


In [ ]:
def save_heatmap(matrix: pd.DataFrame, path: Path, title: str, cmap: str = "viridis", center: float | None = None, fmt: str = ".3f") -> None:
    plt.figure(figsize=(12, max(4, 0.45 * len(matrix))))
    sns.heatmap(matrix, annot=True, fmt=fmt, cmap=cmap, center=center, linewidths=0.5, linecolor="white")
    plt.xlabel("Test dataset / metric")
    plt.ylabel("Strategy")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.show()


save_heatmap(strategy_matrix, OUT_DIR / "strategy_matrix_heatmap.png", "MMseqs-enhanced ceiling-aware weighting: raw scores")
save_heatmap(normalized_headroom, OUT_DIR / "strategy_normalized_headroom_heatmap.png", "Normalized headroom above best simple baseline")
save_heatmap(strategy_delta_vs_same.drop(index="same_dataset_head"), OUT_DIR / "delta_vs_same_dataset_head_heatmap.png", "Delta vs same-dataset head", cmap="vlag", center=0)
save_heatmap(strategy_delta_vs_validation_selected.drop(index="validation_selected_single_head"), OUT_DIR / "delta_vs_validation_selected_heatmap.png", "Delta vs validation-selected head", cmap="vlag", center=0)

summary_plot = strategy_summary.drop(index="best_individual_test_oracle").reset_index()
plt.figure(figsize=(10, 4.5))
sns.barplot(data=summary_plot, x="strategy", y="mean_delta_vs_same_dataset_head", color="#4C78A8")
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=30, ha="right")
plt.xlabel("Strategy")
plt.ylabel("Mean delta vs same-dataset head")
plt.title("Do enhanced ceiling weights improve over same-dataset heads?")
plt.tight_layout()
plt.savefig(OUT_DIR / "mean_delta_vs_same_dataset_head.png", dpi=200)
plt.show()

weighted_only = weights_df[weights_df["strategy"].str.contains("weighted")].copy()
for strategy in weighted_only["strategy"].unique():
    matrix = weighted_only[weighted_only["strategy"] == strategy].set_index("target_dataset")[DATASETS].reindex(index=DATASETS, columns=DATASETS)
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap="mako", linewidths=0.5, linecolor="white")
    plt.xlabel("UdonPred training head")
    plt.ylabel("Target dataset")
    plt.title(f"Weights: {strategy}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"weights_{strategy}.png", dpi=200)
    plt.show()


## 9. Comparison With Previous Ensemble Methods

This section compares the new MMseqs/exact/biological-prior weighting strategies against the earlier ensemble methods from `09_ensembles_ceiling_headroom` / `10`, including convex validation weighting, ceiling-aware validation weighting, and ridge stacking. This is the direct comparison needed to answer whether the enhanced ceiling information improves over the broader ensemble toolbox.


In [ ]:
PREVIOUS_ENSEMBLE_DIR = RESULTS / "ensembles_ceiling_selected_head"
previous_matrix_path = PREVIOUS_ENSEMBLE_DIR / "ensemble_matrix.csv"
if not previous_matrix_path.exists():
    raise FileNotFoundError(f"Missing previous ensemble matrix: {previous_matrix_path}")

previous_matrix = pd.read_csv(previous_matrix_path).set_index("strategy")[METRIC_COLS].apply(pd.to_numeric, errors="coerce")
previous_rows_to_add = [
    "ceiling_aware_selected_single_head",
    "global_convex_validation",
    "per_target_convex_validation",
    "per_target_ceiling_aware_validation",
    "global_ridge_stacking_validation",
    "per_target_ridge_stacking_validation",
]
previous_rows_to_add = [row for row in previous_rows_to_add if row in previous_matrix.index]

combined_matrix = pd.concat(
    [
        strategy_matrix,
        previous_matrix.loc[[row for row in previous_rows_to_add if row not in strategy_matrix.index]],
    ],
    axis=0,
).loc[lambda df: ~df.index.duplicated(keep="first")]

combined_delta_vs_best = combined_matrix.subtract(combined_matrix.loc["best_individual_test_oracle"], axis="columns")
combined_delta_vs_same = combined_matrix.subtract(combined_matrix.loc["same_dataset_head"], axis="columns")
combined_delta_vs_validation = combined_matrix.subtract(combined_matrix.loc["validation_selected_single_head"], axis="columns")
combined_raw_headroom = combined_matrix.subtract(best_simple_baseline_scores, axis="columns")
combined_normalized_headroom = combined_raw_headroom.divide(available_headroom, axis="columns")
combined_normalized_headroom.loc[:, available_headroom <= 1e-12] = np.nan

combined_summary_rows = []
for strategy in combined_matrix.index:
    deltas_best = combined_delta_vs_best.loc[strategy]
    deltas_same = combined_delta_vs_same.loc[strategy]
    deltas_validation = combined_delta_vs_validation.loc[strategy]
    combined_summary_rows.append({
        "strategy": strategy,
        "mean_score": float(combined_matrix.loc[strategy].mean()),
        "mean_normalized_headroom": float(combined_normalized_headroom.loc[strategy].mean()),
        "mean_delta_vs_best_individual": float(deltas_best.mean()),
        "mean_delta_vs_same_dataset_head": float(deltas_same.mean()),
        "mean_delta_vs_validation_selected": float(deltas_validation.mean()),
        "wins_vs_best_individual": int((deltas_best > 1e-6).sum()),
        "ties_vs_best_individual": int((deltas_best.abs() <= 1e-6).sum()),
        "losses_vs_best_individual": int((deltas_best < -1e-6).sum()),
        "wins_vs_same_dataset_head": int((deltas_same > 1e-6).sum()),
        "wins_vs_validation_selected": int((deltas_validation > 1e-6).sum()),
        "worst_delta_vs_same_dataset_head": float(deltas_same.min()),
    })
combined_summary = pd.DataFrame(combined_summary_rows).set_index("strategy").sort_values("mean_score", ascending=False)

combined_matrix.to_csv(OUT_DIR / "combined_previous_and_mmseqs_strategy_matrix.csv")
combined_delta_vs_best.to_csv(OUT_DIR / "combined_delta_vs_best_individual.csv")
combined_delta_vs_same.to_csv(OUT_DIR / "combined_delta_vs_same_dataset_head.csv")
combined_delta_vs_validation.to_csv(OUT_DIR / "combined_delta_vs_validation_selected.csv")
combined_normalized_headroom.to_csv(OUT_DIR / "combined_normalized_headroom.csv")
combined_summary.to_csv(OUT_DIR / "combined_strategy_summary.csv")

display(combined_matrix.style.format("{:.3f}"))
display(combined_normalized_headroom.style.format("{:.3f}"))
display(combined_summary.style.format("{:.3f}"))


### Best Method Per Metric Across All Ensemble Strategies

This table includes the old validation-fitted methods and the new MMseqs/exact/biological-prior weighting methods. The oracle is excluded from `best_non_oracle_strategy`.


In [ ]:
combined_non_oracle = combined_matrix.drop(index="best_individual_test_oracle")
combined_winner_rows = []
for metric in METRIC_COLS:
    best_strategy = combined_non_oracle[metric].idxmax()
    combined_winner_rows.append({
        "metric": metric,
        "best_non_oracle_strategy": best_strategy,
        "best_non_oracle_score": float(combined_non_oracle.loc[best_strategy, metric]),
        "best_individual_oracle_score": float(combined_matrix.loc["best_individual_test_oracle", metric]),
        "same_dataset_score": float(combined_matrix.loc["same_dataset_head", metric]),
        "validation_selected_score": float(combined_matrix.loc["validation_selected_single_head", metric]),
        "delta_best_non_oracle_vs_same": float(combined_non_oracle.loc[best_strategy, metric] - combined_matrix.loc["same_dataset_head", metric]),
        "delta_best_non_oracle_vs_validation": float(combined_non_oracle.loc[best_strategy, metric] - combined_matrix.loc["validation_selected_single_head", metric]),
    })
combined_winners = pd.DataFrame(combined_winner_rows)
combined_winners.to_csv(OUT_DIR / "combined_best_non_oracle_strategy_by_metric.csv", index=False)
display(combined_winners.style.format({
    "best_non_oracle_score": "{:.3f}",
    "best_individual_oracle_score": "{:.3f}",
    "same_dataset_score": "{:.3f}",
    "validation_selected_score": "{:.3f}",
    "delta_best_non_oracle_vs_same": "{:+.3f}",
    "delta_best_non_oracle_vs_validation": "{:+.3f}",
}))


### Combined Comparison Plots


In [ ]:
comparison_order = [
    "best_individual_test_oracle",
    "same_dataset_head",
    "validation_selected_single_head",
    "ceiling_aware_selected_single_head",
    "per_target_convex_validation",
    "per_target_ridge_stacking_validation",
    "exact_ceiling_weighted",
    "mmseqs95_ceiling_weighted",
    "mmseqs80_ceiling_weighted",
    "blended_exact_mmseqs_bio_weighted",
    "per_target_ceiling_aware_validation",
    "simple_mean_all_heads",
    "global_convex_validation",
    "global_ridge_stacking_validation",
]
comparison_order = [row for row in comparison_order if row in combined_matrix.index]

save_heatmap(
    combined_matrix.loc[comparison_order],
    OUT_DIR / "combined_strategy_matrix_heatmap.png",
    "All ensemble strategies: raw scores",
)
save_heatmap(
    combined_normalized_headroom.loc[comparison_order],
    OUT_DIR / "combined_normalized_headroom_heatmap.png",
    "All ensemble strategies: normalized headroom",
)
save_heatmap(
    combined_delta_vs_same.loc[[row for row in comparison_order if row != "same_dataset_head"]],
    OUT_DIR / "combined_delta_vs_same_dataset_head_heatmap.png",
    "All ensemble strategies: delta vs same-dataset head",
    cmap="vlag",
    center=0,
)

plot_summary = combined_summary.reset_index()
plot_summary["strategy"] = pd.Categorical(plot_summary["strategy"], categories=comparison_order, ordered=True)
plot_summary = plot_summary.sort_values("strategy")
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_summary, x="strategy", y="mean_delta_vs_same_dataset_head", color="#4C78A8")
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=35, ha="right")
plt.xlabel("Strategy")
plt.ylabel("Mean delta vs same-dataset head")
plt.title("Previous ensemble methods vs MMseqs-enhanced ceiling weights")
plt.tight_layout()
plt.savefig(OUT_DIR / "combined_mean_delta_vs_same_dataset_head.png", dpi=200)
plt.show()


### Focused Mean Headroom Plot

This plot mirrors the earlier ensemble notebook: it compresses each strategy to mean normalized headroom above the best simple baseline, so the comparison answers how much baseline-to-ceiling room each ensemble captures on average.


In [ ]:
FOCUSED_STRATEGY_LABELS = {
    "best_individual_test_oracle": "Best individual\n(test oracle)",
    "validation_selected_single_head": "Validation-selected\nsingle head",
    "per_target_convex_validation": "Validation-fit\nconvex",
    "per_target_ceiling_aware_validation": "Old broad\ncompatibility weights",
    "per_target_ridge_stacking_validation": "Validation-fit\nridge",
    "exact_ceiling_weighted": "Exact-overlap\nanchored weights",
    "mmseqs95_ceiling_weighted": "Homology 95%\nanchored weights",
    "mmseqs80_ceiling_weighted": "Homology 80%\nanchored weights",
    "blended_exact_mmseqs_bio_weighted": "Blended evidence\nanchored weights",
}

focused_order = [
    "best_individual_test_oracle",
    "validation_selected_single_head",
    "per_target_convex_validation",
    "per_target_ceiling_aware_validation",
    "per_target_ridge_stacking_validation",
    "exact_ceiling_weighted",
    "mmseqs95_ceiling_weighted",
    "mmseqs80_ceiling_weighted",
    "blended_exact_mmseqs_bio_weighted",
]
focused_order = [strategy for strategy in focused_order if strategy in combined_summary.index]
focused_plot = combined_summary.loc[focused_order].reset_index()
focused_plot["label"] = focused_plot["strategy"].map(FOCUSED_STRATEGY_LABELS)

plt.figure(figsize=(12, 5))
sns.barplot(data=focused_plot, x="label", y="mean_normalized_headroom", color="#5B7FA3")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Strategy")
plt.ylabel("Mean normalized headroom")
plt.title("How much baseline-to-ceiling headroom each ensemble captures")
plt.xticks(rotation=0, ha="center")
plt.ylim(0, max(0.58, float(focused_plot["mean_normalized_headroom"].max()) + 0.04))
plt.tight_layout()
plt.savefig(OUT_DIR / "focused_mean_normalized_headroom_with_mmseqs.png", dpi=200)
plt.show()

focused_plot[["strategy", "label", "mean_score", "mean_normalized_headroom", "mean_delta_vs_same_dataset_head"]].to_csv(
    OUT_DIR / "focused_mean_normalized_headroom_with_mmseqs.csv",
    index=False,
)
display(focused_plot[["strategy", "mean_score", "mean_normalized_headroom", "mean_delta_vs_same_dataset_head"]].style.format({
    "mean_score": "{:.3f}",
    "mean_normalized_headroom": "{:.3f}",
    "mean_delta_vs_same_dataset_head": "{:+.3f}",
}))


### Combined Comparison Interpretation

The previous validation-fitted methods remain the strongest way to gain small per-metric improvements. In particular, per-target convex and per-target ridge can improve selected metrics because they directly fit weights on validation predictions. The MMseqs-enhanced ceiling strategies are more biologically constrained, but they are not generally stronger on raw score.

This means the enhanced ceiling analysis is best used as a prior or interpretability layer, not as a replacement for validation-fitted stacking/convex optimization. It helps identify biologically plausible auxiliary heads, but the previous learned ensemble methods are better when the goal is purely to maximize validation/test performance.


## 10. Overall Interpretation

The enhanced ceiling information is useful mainly as a *controlled prior*, not as a replacement for validation performance or same-dataset specificity. The same-dataset head remains a very strong baseline because each UdonPred head is trained to reproduce one annotation definition. This matches the lecture framing: disorder labels are not interchangeable, and the biological meaning of the target matters.

The exact-ceiling strategy is conservative because many exact overlaps are missing or tiny. It can support biologically obvious transfers such as TriZOD/CheZOD or SoftDis/DisProt, but it should not be expected to rescue all targets.

The MMseqs strategies are denser because they use homologous aligned positions. This can reveal useful compatibility that exact overlap misses, especially for pLDDT/DisProt and NMR-related pairs. However, MMseqs ceilings estimate homologous annotation transfer, not direct same-protein reproducibility. They should therefore be downweighted relative to exact evidence.

The blended strategy is the most biologically defensible: exact evidence has highest trust, high-identity MMseqs has intermediate trust, MMseqs-80 is lower trust, and annotation-family priors only regularize missing or sparse measurements. The same-dataset anchor prevents the ensemble from spreading too much weight onto heads that are biologically related but not target-identical.

If the weighted strategies do not beat same-dataset or validation-selected heads, that is still an informative result. It means the enhanced ceiling analysis improves interpretation and weight plausibility, but the available UdonPred heads are already highly target-specific. In that case, ceiling-aware weights are better used for explaining which auxiliary heads are biologically safe to include, rather than for expecting universal raw-performance gains.
